In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

BASE_IMAGE_DIR = "/kaggle/input/idrid1/idrid/segmentation/Original_Images"
BASE_MASK_DIR = "/kaggle/input/idrid1/idrid/segmentation/All_Segmentation_Groundtruths"

LESION_FOLDER = "Hard_Exudates"

OUT_DIR = "/kaggle/input/idrid1/idrid/processed"

IMG_SIZE = 512

In [ ]:
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    final = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)
    return final

In [ ]:
def crop_circle(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        return img[y:y+h, x:x+w]
    return img

In [ ]:
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    img = crop_circle(img)
    img = apply_clahe(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img

In [ ]:
def preprocess_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    mask = (mask > 10).astype(np.uint8) * 255
    return mask

In [ ]:
def process_split(split_name):
    print(f"\nProcessing {split_name} set...")

    img_dir = os.path.join(BASE_IMAGE_DIR, f"{'Training_Set' if split_name=='train' else 'Testing_Set'}")
    mask_dir = os.path.join(BASE_MASK_DIR, f"{'Training_Set' if split_name=='train' else 'Testing_Set'}", LESION_FOLDER)

    save_img_dir = os.path.join(OUT_DIR, split_name, "images")
    save_mask_dir = os.path.join(OUT_DIR, split_name, "masks")

    os.makedirs(save_img_dir, exist_ok=True)
    os.makedirs(save_mask_dir, exist_ok=True)

    image_files = sorted(os.listdir(img_dir))
    mask_files = sorted(os.listdir(mask_dir))

    mask_map = {}
    for f in mask_files:
        base = os.path.splitext(f)[0].split('_')[0] + "_" + os.path.splitext(f)[0].split('_')[1]
        mask_map[base] = f

    count = 0
    for file in tqdm(image_files):
        base = os.path.splitext(file)[0]
        if base not in mask_map:
            continue

        img_path = os.path.join(img_dir, file)
        mask_path = os.path.join(mask_dir, mask_map[base])

        img = preprocess_image(img_path)
        mask = preprocess_mask(mask_path)

        cv2.imwrite(os.path.join(save_img_dir, file), img)
        cv2.imwrite(os.path.join(save_mask_dir, file), mask)

        count += 1

    print(f"Processed {count} image-mask pairs for {split_name} set.")

In [ ]:
if __name__ == "__main__":
    process_split("train")
    process_split("test")
    print("All preprocessing done.")

In [ ]:
!pip install segmentation-models-pytorch
!pip install efficientnet-pytorch

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchvision import transforms

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 30
LR = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DATA_PATH = "/kaggle/input/idrid1/idrid/processed"
SAVE_PATH = "/kaggle/working/unet_exudates.pt"

class RetinalSegDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_files = sorted(os.listdir(image_dir))
        self.transform_img = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])
        self.transform_mask = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_files[idx])
        mask_path = os.path.join(self.mask_dir, self.image_files[idx])

        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))

        img = self.transform_img(img)
        mask = self.transform_mask(mask)
        mask = (mask > 0).float()

        return img, mask

In [ ]:
train_dataset = RetinalSegDataset(
    os.path.join(DATA_PATH, "train", "images"),
    os.path.join(DATA_PATH, "train", "masks")
)
test_dataset = RetinalSegDataset(
    os.path.join(DATA_PATH, "test", "images"),
    os.path.join(DATA_PATH, "test", "masks")
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation='sigmoid'
).to(DEVICE)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - ((2. * intersection + smooth) / (pred.sum() + target.sum() + smooth))

In [ ]:
best_dice = 0.0
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

        preds = model(imgs)
        loss = 0.5 * criterion(preds, masks) + 0.5 * dice_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    model.eval()
    dice_score = 0.0
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            dice_score += (1 - dice_loss(preds, masks)).item()

    avg_dice = dice_score / len(test_loader)

    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Dice={avg_dice:.4f}")

    if avg_dice > best_dice:
        best_dice = avg_dice
        torch.save(model.state_dict(), SAVE_PATH)
        print("Model saved.")

In [ ]:
def show_prediction():
    model.eval()
    imgs, masks = next(iter(test_loader))
    imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

    with torch.no_grad():
        preds = model(imgs)

    img_np = imgs[0].cpu().permute(1,2,0).numpy()
    mask_np = masks[0].cpu().squeeze().numpy()
    pred_np = preds[0].cpu().squeeze().numpy()
    pred_np = (pred_np > 0.4).astype(np.uint8)

    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    plt.imshow((img_np * 0.5 + 0.5))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(mask_np, cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(pred_np, cmap='gray')
    plt.title("Prediction")
    plt.axis("off")
    plt.show()

In [ ]:
show_prediction()